### tokenize(): lexical analysis — turn one expression string into a list of tokens
- scans the string left to right, one character at a time.
- produces `(type, value)` tuples for the parser: `NUM`, `OP`, `LPAREN`, `RPAREN`.
- a number literal is one or more digits, with an optional single `.` followed by one or more digits.
- the minus sign is kept as its own `OP` token (not folded into a number), so the parser can treat it as unary.
- raises `ValueError` on an unexpected character or a malformed number such as `3.`; the caller reports `ERROR`.

In [ ]:
def tokenize(expression: str) -> list:
    """Break one expression string into a list of (type, value) tokens.

    Token types are NUM, OP, LPAREN and RPAREN. The function walks the string
    with an index and groups consecutive digits (and an optional decimal part)
    into a single NUM token. It raises ValueError when it meets a character
    that is not part of the grammar, or a number with a dot but no digits
    after it.
    """
    tokens = []
    i = 0
    length = len(expression)

    while i < length:
        char = expression[i]

        if char == ' ' or char == '\t':          # skip spaces and tabs
            i += 1

        elif char in '+-*/%^':                    # any operator symbol
            tokens.append(("OP", char))
            i += 1

        elif char == '(':
            tokens.append(("LPAREN", "("))
            i += 1

        elif char == ')':
            tokens.append(("RPAREN", ")"))
            i += 1

        elif char.isdigit():                      # read a whole number literal
            start = i
            while i < length and expression[i].isdigit():
                i += 1
            # a number may have a single dot followed by one or more digits
            if i < length and expression[i] == '.':
                i += 1
                if i >= length or not expression[i].isdigit():
                    raise ValueError("a number needs digits after the '.'")
                while i < length and expression[i].isdigit():
                    i += 1
            tokens.append(("NUM", expression[start:i]))

        else:                                     # anything else is not allowed
            raise ValueError("invalid character: " + char)

    tokens.append(("END", ""))
    return tokens

### show the tokens produced for a few sample expressions

In [ ]:
for e in ["3 + 5", "2 + 3 * 4", "-(3 + 4)", "3.5 + 1.5", "3 @ 5"]:
    try:
        print(e, "->", tokenize(e))
    except ValueError as err:
        print(e, "-> ERROR (", err, ")")

3 + 5 -> [('NUM', '3'), ('OP', '+'), ('NUM', '5'), ('END', '')]
2 + 3 * 4 -> [('NUM', '2'), ('OP', '+'), ('NUM', '3'), ('OP', '*'), ('NUM', '4'), ('END', '')]
-(3 + 4) -> [('OP', '-'), ('LPAREN', '('), ('NUM', '3'), ('OP', '+'), ('NUM', '4'), ('RPAREN', ')'), ('END', '')]
3.5 + 1.5 -> [('NUM', '3.5'), ('OP', '+'), ('NUM', '1.5'), ('END', '')]
3 @ 5 -> ERROR ( invalid character: @ )


## Recursive-descent parser

### Parser Function

The `parse()` function converts a list of lexical tokens into a **parse tree (Abstract Syntax Tree)** representing the structure of an arithmetic expression. It uses a recursive-descent parsing approach, where separate helper functions handle different levels of operator precedence.

The parser processes expressions according to the following precedence and associativity rules:

1. **Addition and subtraction (`+`, `-`)** – left-associative.
2. **Multiplication, division, and modulo (`*`, `/`, `%`)** – left-associative. It also supports **implicit multiplication**, where a factor is directly followed by parentheses, such as `2(3 + 4)`.
3. **Unary minus (`-`)** – treated as a prefix operator and supports expressions such as `-5` or `-(2 + 3)`. Unary plus (`+`) is explicitly unsupported.
4. **Exponentiation (`^`)** – right-associative, with the exponent allowed to contain a unary minus, such as `2^-3`.
5. **Primary expressions** – numeric literals and parenthesised expressions.

The helper functions `peek()`, `advance()`, and `expect()` manage token inspection and consumption while tracking the current parsing position. Each parsing function returns a tree node representing the corresponding operation or value.

After parsing the complete expression, the function checks for any remaining tokens. If unexpected tokens, mismatched parentheses, unsupported operators, or other syntax errors are encountered, a `ValueError` is raised. Otherwise, the completed parse tree is returned.

In [ ]:

def parse(tokens: list):
    """Parse a token list into a parse tree. Raises ValueError on any
    syntax error (including leftover tokens after a complete expression).
    """
    pos = 0

    def peek():
        return tokens[pos]

    def advance():
        nonlocal pos
        tok = tokens[pos]
        pos += 1
        return tok

    def expect(token_type):
        nonlocal pos
        tok = tokens[pos]
        if tok[0] != token_type:
            raise ValueError(f"Expected {token_type} but found {tok[0]}")
        pos += 1
        return tok

    # Level 1: + - (left associative)
    def parse_expression():
        node = parse_term()
        while peek()[0] == 'OP' and peek()[1] in ('+', '-'):
            op = advance()[1]
            right = parse_term()
            node = ('binop', op, node, right)
        return node

    # Level 2: * / % and implicit multiplication (left associative)
    def parse_term():
        node = parse_unary()
        while True:
            tok = peek()
            if tok[0] == 'OP' and tok[1] in ('*', '/', '%'):
                op = advance()[1]
                right = parse_unary()
                node = ('binop', op, node, right)
            elif tok[0] == 'LPAREN':
                # Implicit multiplication: a factor directly followed by '('
                right = parse_unary()
                node = ('binop', '*', node, right)
            else:
                break
        return node

    # Level 3: unary minus (prefix). Unary '+' is explicitly unsupported.
    def parse_unary():
        tok = peek()
        if tok[0] == 'OP' and tok[1] == '-':
            advance()
            operand = parse_unary()
            return ('neg', operand)
        if tok[0] == 'OP' and tok[1] == '+':
            raise ValueError("Unary '+' is not supported")
        return parse_power()

    # Level 4: ^ (right associative; exponent may itself carry a unary '-')
    def parse_power():
        base = parse_primary()
        tok = peek()
        if tok[0] == 'OP' and tok[1] == '^':
            advance()
            exponent = parse_unary()
            return ('binop', '^', base, exponent)
        return base

    # Numbers and parenthesised sub-expressions
    def parse_primary():
        tok = peek()
        if tok[0] == 'NUM':
            advance()
            return ('num', float(tok[1]))
        if tok[0] == 'LPAREN':
            advance()
            node = parse_expression()
            expect('RPAREN')
            return node
        raise ValueError(f"Unexpected token {tok[0]}")

    tree = parse_expression()
    if peek()[0] != 'END':
        raise ValueError(f"Unexpected trailing token {peek()[0]}")
    return tree

### Parse Tree Construction & Formatting

The `format_tree()` function converts the parse tree produced by the recursive-descent `parse()` function into the required **formatted parse tree string**.

- It recursively traverses the parse tree from the root to its child nodes.
- A **number node** (`num`) is displayed as its numeric value, for example `5` or `3.5`.
- A **binary operation node** (`binop`) is displayed in the format `(op left right)`, where the operator appears first, followed by the left and right sub-trees.
- A **unary negation node** (`neg`) is displayed in the format `(neg operand)`.
- Nested expressions are formatted recursively, preserving the operator precedence and structure produced by the parser.
- Implicit multiplication is already represented by the parser as a `*` binary operation, so it is displayed as `*` in the parse tree.
- The formatted tree is returned as a string so it can be used in the final output.

In [ ]:
def format_number(value):
    """Format a number for display in the parse tree."""
    value = float(value)

    if value.is_integer():
        return str(int(value))

    return f"{value:.4f}".rstrip('0').rstrip('.')


In [ ]:
def format_tree(node):
    """
    Convert the parse tree into the required string format.

    Tree formats:
        ('num', value) -> value
        ('neg', operand) -> (neg operand)
        ('binop', op, left, right) -> (op left right)
    """

    node_type = node[0]

    # Number literal
    if node_type == 'num':
        return format_number(node[1])

    # Unary negation
    if node_type == 'neg':
        operand = format_tree(node[1])
        return f"(neg {operand})"

    # Binary operation
    if node_type == 'binop':
        operator = node[1]
        left = format_tree(node[2])
        right = format_tree(node[3])

        return f"({operator} {left} {right})"

    # Invalid/unknown tree node
    raise ValueError("Invalid parse tree node")


In [ ]:
def build_parse_tree(tokens):
    """
    Construct and format the parse tree from the token list.

    The parse() function supplied by the parser constructs
    the tree, while format_tree() converts it to the required
    output representation.
    """
    tree = parse(tokens)
    return format_tree(tree)

# Evaluation and Result Formatting

The `eval_tree()` function walks the parse tree produced by `parse()` and computes its numeric
value, applying each operator recursively from the leaves up.

* A `num` node returns its stored value directly.
* A `neg` node returns the negation of its evaluated operand.
* A `binop` node evaluates both sides first, then applies the operator: `+`, `-`, `*`, `/`, `%`, `^`.
* Division and modulo by zero raise `ZeroDivisionError` rather than crashing silently, so the
  caller can report `ERROR` for that expression without stopping the whole file.

`tokens_to_str()` renders the raw token list from `tokenize()` into the required
`[TYPE:value] ... [END]` display format, matching how `format_tree()` renders the parse tree.

`format_result()` applies the required display rule: whole-number results are shown with no
decimal point (e.g. `8`), otherwise the value is rounded to 4 decimal places.

In [ ]:
def tokens_to_str(tokens: list) -> str:
    """Render Rajesh's token list as '[TYPE:value] ... [END]'."""
    parts = []
    for t_type, value in tokens:
        parts.append("[END]" if t_type == "END" else f"[{t_type}:{value}]")
    return " ".join(parts)


def eval_tree(node) -> float:
    """Recursively evaluate a parse tree produced by parse(). Raises
    ZeroDivisionError on division/modulo by zero."""
    kind = node[0]
    if kind == 'num':
        return node[1]
    if kind == 'neg':
        return -eval_tree(node[1])
    if kind == 'binop':
        _, op, left, right = node
        l, r = eval_tree(left), eval_tree(right)
        if op == '+': return l + r
        if op == '-': return l - r
        if op == '*': return l * r
        if op == '/':
            if r == 0:
                raise ZeroDivisionError("division by zero")
            return l / r
        if op == '%':
            if r == 0:
                raise ZeroDivisionError("division by zero")
            return l % r
        if op == '^':
            return l ** r
    raise ValueError("unknown node type")


def format_result(value: float) -> str:
    """Whole-number results print with no decimal point; otherwise
    rounded to 4 decimal places."""
    rounded = round(value, 4)
    if rounded == int(rounded):
        return str(int(rounded))
    return f"{rounded:.4f}".rstrip('0').rstrip('.')

# evaluate_file(): read expressions, run the full pipeline, and write output.txt

* opens `input_path` with `with open(...)` and reads it one expression per line.
* for each line, calls `tokenize()`, then `parse()`, then `format_tree()` (Ashok's and Aryan's
  functions) to get the tokens and tree strings, and `eval_tree()` to compute the result.
* if `tokenize()` raises a `ValueError`, both `tree` and `tokens` are recorded as `ERROR` and the
  line is skipped early — the expression never reaches the parser.
* if `parse()` raises a `ValueError` (tokens were valid but the syntax wasn't), only `tree`
  becomes `ERROR`; the tokens are still shown, matching the required output format.
* if evaluation raises (e.g. division by zero), only `result` becomes `ERROR`; the tree and
  tokens still print normally, since parsing succeeded.
* builds each four-line block with `_format_block()` and joins them with a blank line between
  expressions, exactly as the sample output requires.
* writes the combined text to `output.txt` in the same folder as `input_path`.
* returns a list of dictionaries (one per expression) as required by the `evaluate_file`
  interface in the assignment spec.

In [ ]:
def _format_block(entry: dict) -> str:
    return (f"Input: {entry['input']}\n"
            f"Tree: {entry['tree']}\n"
            f"Tokens: {entry['tokens']}\n"
            f"Result: {entry['result_str']}")


def evaluate_file(input_path: str) -> list:
    import os

    with open(input_path, 'r') as f:
        lines = f.read().splitlines()

    results, blocks = [], []

    for expr in lines:
        entry = {"input": expr, "tree": None, "tokens": None,
                  "result": "ERROR", "result_str": "ERROR"}
        try:
            tokens = tokenize(expr)
            entry["tokens"] = tokens_to_str(tokens)
        except ValueError:
            entry["tokens"] = "ERROR"
            entry["tree"] = "ERROR"
            results.append(entry); blocks.append(_format_block(entry))
            continue

        try:
            tree = parse(tokens)
            entry["tree"] = format_tree(tree)  # Aryan's function
        except ValueError:
            entry["tree"] = "ERROR"
            results.append(entry); blocks.append(_format_block(entry))
            continue

        try:
            value = eval_tree(tree)
            entry["result"] = value
            entry["result_str"] = format_result(value)
        except (ZeroDivisionError, ValueError, OverflowError):
            entry["result"] = "ERROR"
            entry["result_str"] = "ERROR"

        results.append(entry); blocks.append(_format_block(entry))

    out_dir = os.path.dirname(input_path) or "."
    with open(os.path.join(out_dir, "output.txt"), 'w') as f:
        f.write("\n\n".join(blocks) + "\n")

    return [{"input": e["input"], "tree": e["tree"], "tokens": e["tokens"], "result": e["result"]}
            for e in results]

In [ ]:
results = evaluate_file("../Assignment 2 Description/sample_input.txt")
for r in results:
    print(r)